In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/abhayayare/e-commerce-dataset/ecommerce_dataset/products.csv
/kaggle/input/datasets/abhayayare/e-commerce-dataset/ecommerce_dataset/users.csv
/kaggle/input/datasets/abhayayare/e-commerce-dataset/ecommerce_dataset/order_items.csv
/kaggle/input/datasets/abhayayare/e-commerce-dataset/ecommerce_dataset/reviews.csv
/kaggle/input/datasets/abhayayare/e-commerce-dataset/ecommerce_dataset/orders.csv
/kaggle/input/datasets/abhayayare/e-commerce-dataset/ecommerce_dataset/events.csv


In [2]:
import sqlite3

# ECサイトにおける消費者行動分析と商品推薦の提案

## 分析目的

本分析では、ECサイトにおけるユーザー行動データを用いて、消費者がどのような商品に興味を持ち、どのような条件で購買に至るのかを分析する。

最終的には、分析結果をもとに「どのような商品が売れやすいのか」を整理し、ユーザーに対する商品推薦や、ECサイト運営における改善提案につなげることを目的とする。

---

# 現時点での分析方針

## 1. ユーザーの興味・購買傾向を把握する

現時点では、ユーザーの興味を直接表す情報が限定的であるため、以下の特徴量を利用して傾向を分析する。

### 使用を想定している特徴量

| 特徴量 | 想定する意味 |
|---|---|
| 購買数 | 実際の需要・人気 |
| レビュー評価 | 商品満足度 |
| 価格 | 購買ハードル |
| ウィッシュリスト追加数 | 興味・比較検討段階 |
| 閲覧数(view) | 商品への関心 |
| カート追加(cart) | 購買意欲 |

---

# 仮説

現段階では、

> 「購買数とレビュー評価がともに高い商品は、人気商品である」

という仮説を置いて分析を進める。

まずは、この仮説をもとに上位商品を確認し、どのようなカテゴリ・価格帯・特徴を持つ商品が人気なのかを観察する。

---

# 分析ロードマップ

## Phase1 : 人気商品の把握

まずは以下を確認する。

- 購買数上位の商品
- レビュー評価の高い商品
- 購買数と評価の両方が高い商品
- 上位10商品の特徴

ここでは、人気商品の傾向を把握することを目的とする。

---

## Phase2 : 興味・購買行動の分析

ユーザー行動を以下のような段階として整理する。

| 行動 | 意味 |
|---|---|
| view | 興味 |
| wishlist | 比較・検討 |
| cart | 購買意欲 |
| purchase | 実購入 |
| review | 満足度 |

これらをもとに、

- 興味は高いが購入されない商品
- 購入率の高い商品
- 特定カテゴリの行動傾向

などを分析する。

---

## Phase3 : 売れる商品の条件分析

人気商品や購買率の高い商品の特徴を整理し、

- 価格帯
- レビュー傾向
- カテゴリ
- ユーザー行動

などから、「売れやすい商品の条件」を分析する。

---

## Phase4 : 商品推薦・改善提案

分析結果をもとに、

- ユーザーごとの商品推薦
- 類似商品の提案
- 人気商品の特徴整理
- ECサイト改善案

などにつなげることを目指す。

---

# 現時点で重視していること

本分析では、単純な売上集計だけではなく、

> 「ユーザーがどのような行動を経て購入に至るのか」

という消費者行動の流れを重視して分析を進める。

また、分析前の段階として、

- どの特徴量を使用するか
- 各行動をどのように定義するか
- どの仮説を置くか

といった分析設計も重視する。

# 続き


## 目的

ECサイトのユーザー行動データを用いて、
商品がどのような条件で購買されるのかを分析する。

特に以下の流れを明らかにすることを目的とする：

- 興味（view / wishlist）
- 検討（cart）
- 購買（purchase）

この一連の行動構造を商品単位で可視化する。

## アプローチ

データは複数テーブルに分かれているため、
SQL（DuckDB）を用いて商品単位の分析テーブルを構築する。

Pythonでは以下の役割で処理を分担：

- SQL：データ統合・集計（前処理）
- pandas：確認・分析
- 可視化：傾向の把握

## 前処理（SQL）

products、order_items、reviews、eventsを統合し、
商品単位で以下の指標を作成した：

- purchase_count（購買数）
- avg_rating（レビュー評価）
- view_count（閲覧数）
- wishlist_count（興味）
- cart_count（購買意欲）

これにより、商品ごとの行動構造を1つのテーブルに統合した。

■ EC商品分析用・前処理SQL

In [3]:
import pandas as pd
import duckdb

In [4]:
products = pd.read_csv("/kaggle/input/datasets/abhayayare/e-commerce-dataset/ecommerce_dataset/products.csv")
users = pd.read_csv("/kaggle/input/datasets/abhayayare/e-commerce-dataset/ecommerce_dataset/users.csv")
order_items = pd.read_csv("/kaggle/input/datasets/abhayayare/e-commerce-dataset/ecommerce_dataset/order_items.csv")
reviews = pd.read_csv("/kaggle/input/datasets/abhayayare/e-commerce-dataset/ecommerce_dataset/reviews.csv")
orders = pd.read_csv("/kaggle/input/datasets/abhayayare/e-commerce-dataset/ecommerce_dataset/orders.csv")
events = pd.read_csv("/kaggle/input/datasets/abhayayare/e-commerce-dataset/ecommerce_dataset/events.csv")

In [5]:
con = duckdb.connect()

In [6]:
con.register("products", products)
con.register("users", users)
con.register("order_items", order_items)
con.register("reviews", reviews)
con.register("orders", orders)
con.register("events", events)

In [7]:
query = """
SELECT product_id, COUNT(*) AS cnt
FROM order_items
GROUP BY product_id
ORDER BY cnt DESC
"""

con.execute(query).df().head()

,product_id,cnt
0,P000519,38
1,P000762,37
2,P001164,36
3,P000819,35
4,P000902,35


In [8]:
query = """
SELECT
    p.product_id,
    p.category,
    p.price,
    COUNT(oi.order_id) AS cnt
FROM products p
JOIN order_items oi
    ON p.product_id = oi.product_id
GROUP BY p.product_id, p.category, p.price
ORDER BY cnt DESC
"""

con.execute(query).df()

,product_id,category,price,cnt
0,P000519,Electronics,769.71,38
1,P000762,Clothing,185.46,37
2,P001164,Clothing,18.65,36
3,P001354,Toys,96.63,35
4,P000007,Toys,81.17,35
...,...,...,...,...
1995,P001416,Groceries,26.31,10
1996,P001129,Home & Kitchen,305.16,10
1997,P000168,Sports,298.04,10
1998,P001131,Beauty,127.21,9


① カテゴリ別平均購入数

In [9]:
query = """
SELECT
    category,
    CASE
        WHEN price < 50 THEN 'low'
        WHEN price < 200 THEN 'mid'
        ELSE 'high'
    END AS price_range,
    AVG(cnt) AS avg_purchase
FROM (
    SELECT
        p.product_id,
        p.category,
        p.price,
        COUNT(oi.order_id) AS cnt
    FROM products p
    JOIN order_items oi
        ON p.product_id = oi.product_id
    GROUP BY p.product_id, p.category, p.price
) t
GROUP BY category, price_range
ORDER BY avg_purchase DESC;
"""

con.execute(query).df()

,category,price_range,avg_purchase
0,Toys,high,29.000000
1,Automotive,low,23.857143
2,Books,mid,23.325000
3,Home & Kitchen,low,22.583333
4,Pet Supplies,low,22.537736
5,Sports,mid,22.437500
6,Toys,low,22.391753
7,Beauty,low,22.360656
8,Beauty,high,22.166667
9,Pet Supplies,mid,22.075472


② 価格帯ごとの分析

In [10]:
query = """
SELECT
    category,
    CASE
        WHEN price < 50 THEN 'low'
        WHEN price < 200 THEN 'mid'
        ELSE 'high'
    END AS price_range,
    AVG(cnt) AS avg_purchase
FROM (
    SELECT
        p.product_id,
        p.category,
        p.price,
        COUNT(oi.order_id) AS cnt
    FROM products p
    JOIN order_items oi
        ON p.product_id = oi.product_id
    GROUP BY p.product_id, p.category, p.price
) t
GROUP BY category, price_range
ORDER BY avg_purchase DESC;
"""

con.execute(query).df()

,category,price_range,avg_purchase
0,Toys,high,29.000000
1,Automotive,low,23.857143
2,Books,mid,23.325000
3,Home & Kitchen,low,22.583333
4,Pet Supplies,low,22.537736
5,Sports,mid,22.437500
6,Toys,low,22.391753
7,Beauty,low,22.360656
8,Beauty,high,22.166667
9,Pet Supplies,mid,22.075472


## 初期結果

分析の結果、以下の傾向が見られた：

- Toysカテゴリは価格に関係なく購買数が高い
- 多くのカテゴリは価格帯による差が小さい
- Electronicsは比較検討型の行動構造の可能性がある

単純な価格ではなく、カテゴリごとに購買構造が異なる可能性がある。

## 今後の分析

- view → purchaseの転換率分析
- カテゴリ別購買構造の比較
- 価格帯と購買の関係分析
- レコメンドモデルの検討